# Poker Arena — servidor VLM remoto endurecido

Este notebook sobe um **Qwen3-VL-4B experimental** no Colab atrás de um túnel Cloudflare **nomeado**, com hostname estável, TLS e Bearer token. O VLM não é tratado como leitor universal: sua proposta continua com confiança `0.0` no backend e, portanto, não autoriza decisão.

## Pré-requisitos de segurança

Crie um túnel remotamente gerenciado no Cloudflare e configure um hostname HTTPS para encaminhar a `http://127.0.0.1:8080`. No Colab, cadastre três secrets, sem colocá-los no notebook:

- `CLOUDFLARE_TUNNEL_TOKEN`: token do túnel nomeado;
- `POKER_VLM_API_TOKEN`: segredo aleatório com pelo menos 32 caracteres, usado pelo `llama-server`;
- `POKER_VLM_PUBLIC_BASE_URL`: origem estável, por exemplo `https://vlm.seu-dominio.example`.

O backend também exige `POKER_VLM_ALLOWED_HOSTS`, regiões de redação específicas da UI e consentimento efêmero em cada requisição. Sem qualquer um desses itens, nenhuma captura sai da máquina. O cliente solicita `no-store`, mas a política de retenção do provedor/túnel deve ser auditada pelo operador.

Todos os componentes baixados abaixo têm revisão, filename e SHA-256 fixos. O teste do notebook verifica apenas saúde com dados não sensíveis; screenshots reais devem passar pelo backend, consentimento e hook de redação. A saída bruta dos processos é descartada para não persistir prompts; apenas estados fixos entram em um log temporário limitado, removido pelo cleanup.


In [ ]:
from __future__ import annotations

import hashlib
import os
import shutil
import subprocess
import tempfile
import time
import urllib.parse
import urllib.request
from pathlib import Path

# Imutabilidade: atualizar qualquer valor exige revisar compatibilidade e hashes.
LLAMA_CPP_COMMIT = '86a9c79f866799eb0e7e89c03578ccfbcc5d808e'
MODEL_REPO = 'unsloth/Qwen3-VL-4B-Instruct-GGUF'
MODEL_REVISION = '00c00da0690c4b14b5539b02c4ea5d7c9102b35e'
MODEL_FILENAME = 'Qwen3-VL-4B-Instruct-Q4_K_M.gguf'
MODEL_SHA256 = 'd4dcd426bfba75752a312b266b80fec8136fbaca13c62d93b7ac41fa67f0492b'
MMPROJ_FILENAME = 'mmproj-F16.gguf'
MMPROJ_SHA256 = '1b9f4e92f0fbda14d7d7b58baed86039b8a980fe503d9d6a9393f25c0028f1fc'
MODEL_ALIAS = 'qwen3-vl-4b-instruct-q4-k-m-00c00da'
CLOUDFLARED_VERSION = '2026.5.2'
CLOUDFLARED_SHA256 = '5286698547f03df745adb2355f04c12dde52ef425491e81f433642d695521886'


def _sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        for chunk in iter(lambda: handle.read(8 * 1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()


def _download_verified(url: str, destination: Path, expected_sha256: str) -> None:
    parts = urllib.parse.urlsplit(url)
    allowed_hosts = {'github.com', 'huggingface.co'}
    if (parts.scheme != 'https' or parts.hostname not in allowed_hosts
            or parts.username or parts.password or parts.fragment):
        raise RuntimeError('URL de download fora da allowlist HTTPS')
    if parts.hostname == 'github.com' and parts.query:
        raise RuntimeError('download GitHub nao aceita query')
    if parts.hostname == 'huggingface.co' and parts.query not in {'', 'download=true'}:
        raise RuntimeError('query de download Hugging Face invalida')
    if destination.is_symlink():
        raise RuntimeError('destino de download nao pode ser link')
    if destination.exists() and _sha256(destination) == expected_sha256:
        return
    partial = destination.with_suffix(destination.suffix + '.part')
    partial.unlink(missing_ok=True)
    request = urllib.request.Request(url, headers={'User-Agent': 'poker-arena-colab/1'})  # noqa: S310 - allowlist HTTPS acima
    with urllib.request.urlopen(request, timeout=60) as response, partial.open('wb') as out:  # noqa: S310 - allowlist HTTPS acima
        shutil.copyfileobj(response, out, length=8 * 1024 * 1024)
    actual = _sha256(partial)
    if actual != expected_sha256:
        partial.unlink(missing_ok=True)
        raise RuntimeError(f'hash inválido para {destination.name}: {actual}')
    partial.replace(destination)


def _required_secret(name: str) -> str:
    try:
        from google.colab import userdata
        value = userdata.get(name)
    except Exception as exc:
        raise RuntimeError(f'secret Colab ausente: {name}') from exc
    if not isinstance(value, str) or not value:
        raise RuntimeError(f'secret Colab ausente: {name}')
    return value


def _required_executable(name: str) -> str:
    executable = shutil.which(name)
    if not executable:
        raise RuntimeError(f'executavel obrigatorio ausente: {name}')
    return executable


def _validate_public_base_url(raw: str) -> tuple[str, str]:
    value = raw.rstrip('/')
    parts = urllib.parse.urlsplit(value)
    host = (parts.hostname or '').lower().rstrip('.')
    if parts.scheme != 'https' or not host or parts.username or parts.password:
        raise RuntimeError('POKER_VLM_PUBLIC_BASE_URL exige origem HTTPS sem credenciais')
    if parts.query or parts.fragment or parts.path not in {'', '/'}:
        raise RuntimeError('POKER_VLM_PUBLIC_BASE_URL deve conter apenas a origem')
    if host in {'localhost', '127.0.0.1', '::1'}:
        raise RuntimeError('o hostname público não pode ser loopback')
    return value, host


nvidia_smi = _required_executable('nvidia-smi')
git_executable = _required_executable('git')
cmake_executable = _required_executable('cmake')
tool_env_names = ('PATH', 'HOME', 'TMPDIR', 'LANG', 'LC_ALL', 'TZ', 'LD_LIBRARY_PATH', 'CUDA_VISIBLE_DEVICES', 'NVIDIA_VISIBLE_DEVICES', 'NVIDIA_DRIVER_CAPABILITIES', 'SSL_CERT_FILE', 'SSL_CERT_DIR')
tool_env = {name: os.environ[name] for name in tool_env_names if name in os.environ}
tool_env.update({'GIT_CONFIG_GLOBAL': os.devnull, 'GIT_CONFIG_NOSYSTEM': '1'})
gpu = subprocess.run(  # noqa: S603 - executavel resolvido e argv fixo
    [nvidia_smi, '--query-gpu=name,memory.total', '--format=csv,noheader'],
    capture_output=True,
    text=True,
    check=False,
    env=tool_env,
).stdout.strip()
if not gpu:
    raise RuntimeError('GPU não detectada; selecione um runtime GPU no Colab')
print('GPU:', gpu)

# cloudflared: versão e binário fixos, com verificação antes da execução.
cloudflared = Path('cloudflared')
cloudflared_url = (
    f'https://github.com/cloudflare/cloudflared/releases/download/{CLOUDFLARED_VERSION}/'
    'cloudflared-linux-amd64'
)
_download_verified(cloudflared_url, cloudflared, CLOUDFLARED_SHA256)
cloudflared.chmod(0o700)

# llama.cpp: fonte e build novos a cada execução; nenhum cache binário mutável é confiado.
llama_workspace = Path(tempfile.mkdtemp(prefix='poker-llama-source-')).resolve()
llama_dir = llama_workspace / 'llama.cpp'
llama_dir.mkdir()
subprocess.run([git_executable, '-C', str(llama_dir), 'init'], check=True, env=tool_env)  # noqa: S603 - git resolvido
subprocess.run(  # noqa: S603 - git resolvido e origem fixa
    [git_executable, '-C', str(llama_dir), 'remote', 'add', 'origin',
     'https://github.com/ggml-org/llama.cpp.git'],
    check=True, env=tool_env,
)
subprocess.run(  # noqa: S603 - git resolvido e commit pinado
    [git_executable, '-C', str(llama_dir), 'fetch', '--depth', '1', 'origin', LLAMA_CPP_COMMIT],
    check=True, env=tool_env,
)
subprocess.run(  # noqa: S603 - git resolvido e commit pinado
    [git_executable, '-C', str(llama_dir), 'checkout', '--detach', LLAMA_CPP_COMMIT],
    check=True, env=tool_env,
)
head = subprocess.check_output([git_executable, '-C', str(llama_dir), 'rev-parse', 'HEAD'], text=True, env=tool_env).strip()  # noqa: S603 - git resolvido
if head != LLAMA_CPP_COMMIT:
    raise RuntimeError(f'llama.cpp divergente: {head}')
working_tree = subprocess.check_output(  # noqa: S603 - git resolvido e fonte nova
    [git_executable, '-C', str(llama_dir), 'status', '--porcelain', '--untracked-files=all'],
    text=True, env=tool_env,
).strip()
if working_tree:
    raise RuntimeError('checkout llama.cpp contem alteracoes ou arquivos nao rastreados')

server_binary = llama_dir / 'build' / 'bin' / 'llama-server'
subprocess.run(  # noqa: S603 - cmake resolvido e argumentos controlados
    [cmake_executable, '-S', str(llama_dir), '-B', str(llama_dir / 'build'),
     '-DGGML_CUDA=ON', '-DLLAMA_CURL=OFF'],
    check=True, env=tool_env,
)
subprocess.run(  # noqa: S603 - cmake resolvido e alvo fixo
    [cmake_executable, '--build', str(llama_dir / 'build'), '--config', 'Release',
     '-j', '--target', 'llama-server'],
    check=True, env=tool_env,
)
if not server_binary.is_file() or server_binary.is_symlink():
    raise RuntimeError('llama-server compilado ausente ou link recusado')
server_build_sha256 = _sha256(server_binary)

# Pesos: URLs incluem commit HF e filenames exatos; SHA-256 é o LFS oid publicado.
model_dir = Path('models')
if model_dir.is_symlink():
    raise RuntimeError('diretorio de modelos nao pode ser link')
model_dir.mkdir(exist_ok=True)
model_path = model_dir / MODEL_FILENAME
mmproj_path = model_dir / MMPROJ_FILENAME
hf_base = f'https://huggingface.co/{MODEL_REPO}/resolve/{MODEL_REVISION}'
_download_verified(f'{hf_base}/{MODEL_FILENAME}?download=true', model_path, MODEL_SHA256)
_download_verified(f'{hf_base}/{MMPROJ_FILENAME}?download=true', mmproj_path, MMPROJ_SHA256)

# Processos recebem apenas variáveis operacionais allowlisted e o secret estritamente necessário.
PROCESS_ENV_ALLOWLIST = (
    'PATH', 'HOME', 'TMPDIR', 'LANG', 'LC_ALL', 'TZ', 'LD_LIBRARY_PATH',
    'CUDA_VISIBLE_DEVICES', 'NVIDIA_VISIBLE_DEVICES', 'NVIDIA_DRIVER_CAPABILITIES',
    'SSL_CERT_FILE', 'SSL_CERT_DIR',
)
STATUS_LOG_LIMIT_BYTES = 4096
SAFE_STATUS_MESSAGES = frozenset({
    'server_started', 'server_healthy', 'tunnel_started', 'tunnel_stable',
    'cleanup_started',
})


def _minimal_process_env(extra: dict[str, str]) -> dict[str, str]:
    env = {
        name: value
        for name in PROCESS_ENV_ALLOWLIST
        if (value := os.environ.get(name))
    }
    env.update(extra)
    return env


def _write_status(handle, message: str) -> None:
    # Somente estados fixos: nunca prompts, URLs, exceções ou tokens.
    if message not in SAFE_STATUS_MESSAGES:
        raise ValueError('status de lifecycle não permitido')
    line = f'{message}\n'
    if handle.tell() + len(line.encode('ascii')) <= STATUS_LOG_LIMIT_BYTES:
        handle.write(line)
        handle.flush()


def _wait_health(process: subprocess.Popen[bytes], url: str, timeout: float = 900) -> None:
    parts = urllib.parse.urlsplit(url)
    if (parts.scheme, parts.hostname, parts.port, parts.path, parts.query, parts.fragment) != ('http', '127.0.0.1', 8080, '/health', '', ''):
        raise RuntimeError('health URL deve ser o endpoint loopback fixo')
    deadline = time.monotonic() + timeout
    while time.monotonic() < deadline:
        if process.poll() is not None:
            raise RuntimeError('llama-server encerrou durante a verificação de saúde')
        try:
            with urllib.request.urlopen(url, timeout=3) as response:  # noqa: S310 - URL loopback exata validada acima
                if response.status == 200:
                    return
        except OSError:
            pass
        remaining = deadline - time.monotonic()
        if remaining > 0:
            time.sleep(min(4, remaining))
    raise TimeoutError('llama-server não ficou saudável dentro do limite')


def _wait_process_stable(
    process: subprocess.Popen[bytes], *, name: str, seconds: float
) -> None:
    deadline = time.monotonic() + seconds
    while time.monotonic() < deadline:
        if process.poll() is not None:
            raise RuntimeError(f'{name} encerrou durante a verificação inicial')
        time.sleep(min(0.5, max(0.0, deadline - time.monotonic())))


def _stop_process(process: subprocess.Popen[bytes] | None) -> None:
    if process is None:
        return
    try:
        if process.poll() is None:
            try:
                process.terminate()
            except OSError:
                pass
            try:
                process.wait(timeout=10)
            except subprocess.TimeoutExpired:
                try:
                    process.kill()
                except OSError:
                    pass
                try:
                    process.wait(timeout=5)
                except subprocess.TimeoutExpired:
                    pass
        else:
            process.wait(timeout=1)
    finally:
        for stream in (process.stdin, process.stdout, process.stderr):
            if stream is not None and not stream.closed:
                stream.close()


def _cleanup_runtime(processes: list[subprocess.Popen[bytes]], handles: list, runtime_dir: Path | None) -> None:
    # Idempotente: processos já encerrados, handles fechados e diretório ausente são aceitos.
    for process in reversed(processes):
        try:
            _stop_process(process)
        except (OSError, subprocess.TimeoutExpired):
            pass
    for handle in handles:
        if not handle.closed:
            try:
                handle.close()
            except OSError:
                pass
    if runtime_dir is not None:
        shutil.rmtree(runtime_dir, ignore_errors=True)


processes: list[subprocess.Popen[bytes]] = []
log_handles: list = []
runtime_dir: Path | None = None
status_log = None
server: subprocess.Popen[bytes] | None = None
tunnel: subprocess.Popen[bytes] | None = None
server_env: dict[str, str] = {}
tunnel_env: dict[str, str] = {}

# Segredos só entram depois de downloads, checkout limpo, build e hashes concluídos.
api_token = _required_secret('POKER_VLM_API_TOKEN')
tunnel_token = _required_secret('CLOUDFLARE_TUNNEL_TOKEN')
public_base_url, public_host = _validate_public_base_url(
    _required_secret('POKER_VLM_PUBLIC_BASE_URL')
)
if not 32 <= len(api_token) <= 512 or any(ord(char) < 33 or ord(char) > 126 for char in api_token):
    raise RuntimeError('POKER_VLM_API_TOKEN deve ter 32–512 caracteres ASCII sem espaços')
if any(char.isspace() for char in tunnel_token):
    raise RuntimeError('CLOUDFLARE_TUNNEL_TOKEN tem formato inválido')

try:
    runtime_dir = Path(tempfile.mkdtemp(prefix='poker-vlm-runtime-'))
    status_log = (runtime_dir / 'lifecycle.log').open('w', encoding='ascii')
    log_handles.append(status_log)

    # Saída bruta é descartada: ela pode conter prompts. O log temporário aceita só estados fixos.
    server_env = _minimal_process_env({'LLAMA_API_KEY': api_token})
    server = subprocess.Popen(  # noqa: S603 - binario resolvido e argv fixo
        [str(server_binary.resolve()), '--model', str(model_path.resolve()),
         '--mmproj', str(mmproj_path.resolve()), '--alias', MODEL_ALIAS,
         '--n-gpu-layers', '99', '--host', '127.0.0.1', '--port', '8080',
         '--jinja', '--ctx-size', '8192', '--no-webui', '--no-cache-prompt',
         '--timeout', '30'],
        env=server_env,
        stdin=subprocess.DEVNULL,
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
        close_fds=True,
    )
    processes.append(server)
    server_env.pop('LLAMA_API_KEY', None)
    api_token = ''
    _write_status(status_log, 'server_started')
    _wait_health(server, 'http://127.0.0.1:8080/health')
    _write_status(status_log, 'server_healthy')

    # Túnel nomeado: sem quick tunnel, sem atualização automática e sem token em argv.
    tunnel_env = _minimal_process_env({'TUNNEL_TOKEN': tunnel_token})
    tunnel = subprocess.Popen(  # noqa: S603 - binario verificado por SHA-256 e argv fixo
        [str(cloudflared.resolve()), 'tunnel', '--no-autoupdate', 'run'],
        env=tunnel_env,
        stdin=subprocess.DEVNULL,
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
        close_fds=True,
    )
    processes.append(tunnel)
    tunnel_env.pop('TUNNEL_TOKEN', None)
    tunnel_token = ''
    _write_status(status_log, 'tunnel_started')
    _wait_process_stable(tunnel, name='cloudflared', seconds=8)
    _write_status(status_log, 'tunnel_stable')

    vlm_url = f'{public_base_url}/v1/chat/completions'
    print('Servidor pronto. Configure no backend local (o token não é exibido):')
    print(f'  $env:POKER_VLM_URL = "{vlm_url}"')
    print(f'  $env:POKER_VLM_ALLOWED_HOSTS = "{public_host}"')
    print(f'  $env:POKER_VLM_MODEL = "{MODEL_ALIAS}"')
    print('  $env:POKER_VLM_API_TOKEN = <o mesmo secret, definido fora do código>')
    print('  $env:POKER_VLM_REDACT_REGIONS = <máscaras x0,y0,x1,y1 específicas da UI>')
    print('  $env:POKER_ENABLE_REMOTE_VLM = "1"')
    print('Nenhum screenshot real foi usado no smoke; o backend exige consentimento e redação.')

    while True:
        if server.poll() is not None:
            raise RuntimeError('llama-server encerrou inesperadamente')
        if tunnel.poll() is not None:
            raise RuntimeError('cloudflared encerrou inesperadamente')
        time.sleep(30)
finally:
    server_env.pop('LLAMA_API_KEY', None)
    tunnel_env.pop('TUNNEL_TOKEN', None)
    api_token = ''
    tunnel_token = ''
    if status_log is not None and not status_log.closed:
        try:
            _write_status(status_log, 'cleanup_started')
        except (OSError, ValueError):
            pass
    _cleanup_runtime(processes, log_handles, runtime_dir)
    shutil.rmtree(llama_workspace, ignore_errors=True)
